# Full Rigorous Master Benchmark & Adversarial Audit Notebook
### FICOS Platform — Dry-Bulk Freight Forecasting & Vessel Chartering Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SSOHEB/FICOS-Platform/blob/main/notebooks/colab_freight_forecasting_benchmark.ipynb)

--- 

## Methodological Grounding & Rigorous Design
1. **Baseline Superiority in Maritime Econometrics:** Replicates the empirical finding from **Katris & Kavussanos (2021)** (*Journal of Forecasting*), demonstrating that simple baselines or regularized linear models frequently match or exceed complex non-linear ML models in freight rate forecasting due to high market regime variance.
2. **Risk-Coverage Selective Classification:** Implements the selective-classification framework of **Geifman & El-Yaniv (2017)**, evaluating models strictly on the dual-axis of **Precision** and **Coverage** (rejecting low-conviction signals inside validation noise bands).
3. **Probabilistic-Forecast Procurement:** Adopts the decision-theoretic chartering framework of **Sel & Minner (2022, 2025)**, converting point forecasts into actionable directional procurement commitments (BUY NOW vs. WAIT) gated by empirical validation residual quantiles ($P_{10}, P_{90}$).

---

In [ ]:
# CELL 1: ENVIRONMENT SETUP & DATASET VERIFICATION
import os, sys, subprocess, pandas as pd, numpy as np

print('=' * 80)
print('CELL 1: ENVIRONMENT SETUP & DATASET VERIFICATION')
print('=' * 80)

if 'google.colab' in sys.modules:
    if not os.path.exists('FICOS-Platform'):
        print('Cloning FICOS-Platform repository...')
        subprocess.run(['git', 'clone', 'https://github.com/SSOHEB/FICOS-Platform.git'])
    os.chdir('FICOS-Platform')

ds_path = 'outputs/modeling_dataset.csv'
assert os.path.exists(ds_path), f'Missing dataset file at {ds_path}!'

df = pd.read_csv(ds_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

n_rows = len(df)
n_cols = len(df.columns)
min_date = df['date'].min().strftime('%Y-%m-%d')
max_date = df['date'].max().strftime('%Y-%m-%d')

print(f'Dataset File: {os.path.abspath(ds_path)}')
print(f'Total Rows: {n_rows} | Total Columns: {n_cols}')
print(f'Date Range: {min_date} to {max_date}')

assert n_rows == 2581, f'Expected 2,581 rows, found {n_rows}'
assert min_date == '2016-01-04', f'Expected start 2016-01-04, found {min_date}'
assert max_date == '2026-09-04', f'Expected end 2026-09-04, found {max_date}'
print('✅ CONFIRMED: Dataset matches canonical 2,581 rows (2016-01-04 to 2026-09-04).')


In [ ]:
# CELL 2: PRE-TESTING DIAGNOSTICS & TARGET IMBALANCE AUDIT
print('=' * 80)
print('CELL 2: PRE-TESTING DIAGNOSTICS & TARGET IMBALANCE AUDIT')
print('=' * 80)

assets = ['cape', 'panamax', 'supramax', 'handy', 'kdci']
horizons = [1, 7, 14, 30]

imbalance_report = []
for asset in assets:
    for h in horizons:
        target_col = f'target_{asset}_{h}d'
        if target_col in df.columns:
            deltas = df[target_col] - df[asset]
            n_pos = (deltas > 0).sum()
            n_neg = (deltas < 0).sum()
            n_zero = (deltas == 0).sum()
            pct_pos = (n_pos / len(deltas.dropna())) * 100
            imbalance_report.append({
                'Asset': asset.upper(),
                'Horizon': f'{h}d',
                'Positive Moves (Up)': n_pos,
                'Negative Moves (Down)': n_neg,
                'Flat Moves': n_zero,
                'Up Ratio (%)': round(pct_pos, 1)
            })

df_imb = pd.DataFrame(imbalance_report)
print(df_imb.to_string(index=False))
print('\n✅ Pre-testing target imbalance audit complete.')


In [ ]:
# CELL 3: FEATURE AUDIT & STRICT LEAKAGE QUARANTINE
print('=' * 80)
print('CELL 3: FEATURE AUDIT & STRICT LEAKAGE QUARANTINE')
print('=' * 80)

all_cols = list(df.columns)
leakage_cols = [c for c in all_cols if c.startswith('dir_') or c.startswith('future_') or c.startswith('target_')]
drop_cols = set(leakage_cols + ['date'])
feature_cols = [c for c in all_cols if c not in drop_cols]

print(f'Total Features Analyzed: {len(all_cols)}')
print(f'Target & Directional Leakage Columns Excluded ({len(leakage_cols)} columns):')
for c in sorted(leakage_cols):
    print(f'  - Excluded: {c}')
print(f'\nFinal Clean Predictor Matrix (X): {len(feature_cols)} clean features.')
assert not any(c.startswith('dir_') for c in feature_cols), 'Leakage columns found in predictor matrix!'
print('✅ Leakage audit verified: Zero forward-looking columns present in feature matrix.')


In [ ]:
# CELL 4: CHRONOLOGICAL 70/15/15 SPLIT SETUP
print('=' * 80)
print('CELL 4: CHRONOLOGICAL 70/15/15 SPLIT SETUP')
print('=' * 80)

n_train = int(n_rows * 0.70)
n_val = int(n_rows * 0.15)
n_test = n_rows - n_train - n_val

train_dates = (df['date'].iloc[0].strftime('%Y-%m-%d'), df['date'].iloc[n_train-1].strftime('%Y-%m-%d'))
val_dates   = (df['date'].iloc[n_train].strftime('%Y-%m-%d'), df['date'].iloc[n_train+n_val-1].strftime('%Y-%m-%d'))
test_dates  = (df['date'].iloc[n_train+n_val].strftime('%Y-%m-%d'), df['date'].iloc[-1].strftime('%Y-%m-%d'))

print(f'Train Split ({n_train} rows, 70%): {train_dates[0]} to {train_dates[1]}')
print(f'Val Split   ({n_val} rows, 15%):   {val_dates[0]} to {val_dates[1]}')
print(f'Test Split  ({n_test} rows, 15%):  {test_dates[0]} to {test_dates[1]}')


In [ ]:
# CELL 5: DEEP MODEL ARCHITECTURES DEFINITION (ALL 8 MODELS)
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
import xgboost as xgb, lightgbm as lgb

print('Model Candidates Initialized:')
print('  1. Persistence Baseline (Delta = 0)')
print('  2. Ridge Regression (alpha=10.0)')
print('  3. ElasticNet (alpha=0.1, l1_ratio=0.5)')
print('  4. RandomForest (n_estimators=100, max_depth=5)')
print('  5. XGBoost (n_estimators=100, max_depth=4, lr=0.03)')
print('  6. LightGBM (n_estimators=100, max_depth=4, lr=0.03)')
print('  7. PyTorch GRU (hidden_dim=32, num_layers=2, 30 epochs)')
print('  8. PyTorch LSTM (hidden_dim=32, num_layers=2, 30 epochs)')

class PyTorchDeepGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, num_layers=2):
        super(PyTorchDeepGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

class PyTorchDeepLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, num_layers=2):
        super(PyTorchDeepLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

def train_deep_sequence_model(model_class, X_tr, y_tr, X_va, X_te, seq_len=10, epochs=30, lr=0.005):
    input_dim = X_tr.shape[1]
    def create_seqs(X_data, y_data):
        seqs_X, seqs_y = [], []
        for i in range(seq_len, len(X_data)):
            seqs_X.append(X_data[i-seq_len:i])
            seqs_y.append(y_data[i])
        return torch.tensor(np.array(seqs_X), dtype=torch.float32), torch.tensor(np.array(seqs_y), dtype=torch.float32)

    X_tr_seq, y_tr_seq = create_seqs(X_tr, y_tr)
    model = model_class(input_dim=input_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()
    dataset = TensorDataset(X_tr_seq, y_tr_seq)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    model.train()
    for _ in range(epochs):
        for bx, by in loader:
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        def predict_seq(X_data, y_data):
            if len(X_data) <= seq_len:
                return np.zeros(len(X_data))
            seqs_X, _ = create_seqs(X_data, y_data)
            preds = model(seqs_X).numpy()
            return np.concatenate([np.zeros(seq_len), preds])

        p_tr = predict_seq(X_tr, y_tr)
        p_va = predict_seq(X_va, np.zeros(len(X_va)))
        p_te = predict_seq(X_te, np.zeros(len(X_te)))
        
    return p_tr, p_va, p_te


In [ ]:
# CELL 6: FULL 20-PAIR BENCHMARK TOURNAMENT (REAL TRAINING)
from scipy.stats import beta

def calc_smape(y_true, y_pred):
    return float(np.mean(200 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)))

def calc_delta_r2(delta_true, delta_pred):
    ss_tot = np.sum((delta_true - np.mean(delta_true)) ** 2)
    ss_res = np.sum((delta_true - delta_pred) ** 2)
    return float(1 - ss_res / ss_tot if ss_tot > 0 else np.nan)

def calc_ungated_da(delta_true, delta_pred):
    act = np.sign(delta_true)
    prd = np.sign(delta_pred)
    valid = (act != 0)
    return float(np.mean(act[valid] == prd[valid]) * 100.0) if valid.sum() > 0 else np.nan

def clopper_pearson_ci(k, n, alpha=0.10):
    if n == 0: return (np.nan, np.nan)
    lower = 0.0 if k == 0 else beta.ppf(alpha / 2, k, n - k + 1)
    upper = 1.0 if k == n else beta.ppf(1 - alpha / 2, k + 1, n - k)
    return (float(lower * 100.0), float(upper * 100.0))

assets = ['cape', 'panamax', 'supramax', 'handy', 'kdci']
horizons = [1, 7, 14, 30]
master_results = []

print('Executing Real Benchmark Tournament Across All 20 Pairs...')
for asset in assets:
    for h in horizons:
        horizon_str = f'{h}d'
        pair_title = f'{asset.upper()} {horizon_str}'
        print(f'Training & Evaluating: {pair_title}...')
        
        df_pair = df.copy()
        df_pair['_y_target'] = df_pair[asset].shift(-h)
        df_pair['_y_delta'] = df_pair['_y_target'] - df_pair[asset]
        df_v = df_pair[~df_pair['_y_delta'].isna()].reset_index(drop=True)
        
        tr_len = min(n_train, len(df_v))
        va_len = min(n_val, max(0, len(df_v) - tr_len))
        te_len = max(0, len(df_v) - tr_len - va_len)
        
        tr_m = np.zeros(len(df_v), dtype=bool); tr_m[:tr_len] = True
        va_m = np.zeros(len(df_v), dtype=bool); va_m[tr_len:tr_len+va_len] = True
        te_m = np.zeros(len(df_v), dtype=bool); te_m[tr_len+va_len:] = True
        
        X_raw = df_v[feature_cols].values
        y_delta = df_v['_y_delta'].values
        y_base = df_v[asset].values
        y_target = df_v['_y_target'].values
        
        med = np.nanmedian(X_raw[tr_m], axis=0)
        med = np.where(np.isnan(med), 0.0, med)
        for c_idx in range(X_raw.shape[1]):
            X_raw[:, c_idx] = np.where(np.isnan(X_raw[:, c_idx]), med[c_idx], X_raw[:, c_idx])
            
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_raw[tr_m])
        X_va_s = scaler.transform(X_raw[va_m])
        X_te_s = scaler.transform(X_raw[te_m])
        
        k_best = min(50, X_raw.shape[1])
        sel = SelectKBest(f_regression, k=k_best)
        X_tr_sel = sel.fit_transform(X_tr_s, y_delta[tr_m])
        X_va_sel = sel.transform(X_va_s)
        X_te_sel = sel.transform(X_te_s)
        
        gru_tr, gru_va, gru_te = train_deep_sequence_model(PyTorchDeepGRU, X_tr_sel, y_delta[tr_m], X_va_sel, X_te_sel, epochs=20)
        lstm_tr, lstm_va, lstm_te = train_deep_sequence_model(PyTorchDeepLSTM, X_tr_sel, y_delta[tr_m], X_va_sel, X_te_sel, epochs=20)
        
        models = {
            'Persistence': None,
            'Ridge': Ridge(alpha=10.0),
            'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=1000, random_state=42),
            'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
            'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.03, random_state=42, n_jobs=-1),
            'LightGBM': lgb.LGBMRegressor(n_estimators=100, max_depth=4, learning_rate=0.03, random_state=42, verbose=-1, n_jobs=-1),
            'GRU_seq': (gru_tr, gru_va, gru_te),
            'LSTM_seq': (lstm_tr, lstm_va, lstm_te)
        }
        
        model_scores = {}
        for m_name, m_obj in models.items():
            if m_name == 'Persistence':
                pred_d_tr = np.zeros(tr_len); pred_d_va = np.zeros(va_len); pred_d_te = np.zeros(te_len)
            elif isinstance(m_obj, tuple):
                pred_d_tr, pred_d_va, pred_d_te = m_obj
            else:
                m_obj.fit(X_tr_sel, y_delta[tr_m])
                pred_d_tr = m_obj.predict(X_tr_sel)
                pred_d_va = m_obj.predict(X_va_sel)
                pred_d_te = m_obj.predict(X_te_sel)
                
            sm_tr = calc_smape(y_target[tr_m], y_base[tr_m] + pred_d_tr)
            sm_va = calc_smape(y_target[va_m], y_base[va_m] + pred_d_va)
            sm_te = calc_smape(y_target[te_m], y_base[te_m] + pred_d_te)
            d_r2 = calc_delta_r2(y_delta[te_m], pred_d_te)
            da_te = calc_ungated_da(y_delta[te_m], pred_d_te)
            model_scores[m_name] = {'tr_smape': sm_tr, 'va_smape': sm_va, 'te_smape': sm_te, 'delta_r2': d_r2, 'da_te': da_te, 'pred_d_va': pred_d_va, 'pred_d_te': pred_d_te}
            
        best_m_name = min(model_scores.keys(), key=lambda k: model_scores[k]['va_smape'])
        best_sc = model_scores[best_m_name]
        
        # Uncertainty Gating & Permutation Test & Verdict
        val_resids = y_delta[va_m] - best_sc['pred_d_va']
        p10 = float(np.percentile(val_resids, 10))
        p90 = float(np.percentile(val_resids, 90))
        tau = 0.01
        
        te_y_base = y_base[te_m]; te_y_delta = y_delta[te_m]; te_pred_d = best_sc['pred_d_te']
        pct_pred = te_pred_d / (np.abs(te_y_base) + 1e-8)
        buy_m = (te_pred_d > max(0.0, p90)) & (pct_pred > tau)
        wait_m = (te_pred_d < min(0.0, p10)) & (pct_pred < -tau)
        fired_m = buy_m | wait_m
        n_fired = int(fired_m.sum())
        coverage_pct = float(n_fired / te_len * 100.0) if te_len > 0 else 0.0
        n_corr = int((te_y_delta[buy_m] > 0).sum()) + int((te_y_delta[wait_m] < 0).sum())
        gated_prec = float(n_corr / n_fired * 100.0) if n_fired > 0 else np.nan
        cp_low, cp_high = clopper_pearson_ci(n_corr, n_fired)
        
        perm_das = []
        rng_p = np.random.RandomState(42)
        for _ in range(20):
            r_perm = Ridge(alpha=10.0)
            r_perm.fit(X_tr_sel, rng_p.permutation(y_delta[tr_m]))
            perm_das.append(calc_ungated_da(te_y_delta, r_perm.predict(X_te_sel)))
        perm_noise_max = float(np.max(perm_das))
        passes_perm = bool(best_sc['da_te'] > perm_noise_max)
        
        if n_fired < 20:
            verdict = 'INSUFFICIENT SAMPLE SIZE FOR CONFIDENCE'
        elif not passes_perm:
            verdict = 'FAILS PERMUTATION TEST'
        elif best_sc['tr_smape'] < best_sc['va_smape'] - 5.0:
            verdict = 'OVERFIT'
        elif best_sc['delta_r2'] < -0.20 or best_sc['da_te'] < 50.0:
            verdict = 'UNDERFIT'
        else:
            verdict = 'GENUINE SIGNAL'
            
        master_results.append({
            'asset': asset, 'horizon': horizon_str, 'best_model': best_m_name,
            'train_smape': round(best_sc['tr_smape'], 2), 'val_smape': round(best_sc['va_smape'], 2), 'test_smape': round(best_sc['te_smape'], 2),
            'test_delta_r2': round(best_sc['delta_r2'], 4), 'ungated_da': round(best_sc['da_te'], 1), 'perm_noise_max': round(perm_noise_max, 1),
            'passes_perm': passes_perm, 'gated_precision': round(gated_prec, 1) if not np.isnan(gated_prec) else None,
            'cp_90_low': round(cp_low, 1) if not np.isnan(cp_low) else None, 'cp_90_high': round(cp_high, 1) if not np.isnan(cp_high) else None,
            'gated_coverage': round(coverage_pct, 1), 'n_fired': n_fired, 'verdict': verdict
        })

df_res = pd.DataFrame(master_results)
print('=' * 80)
print('MASTER CONSOLIDATED BENCHMARK TABLE (20 PAIRS)')
print('=' * 80)
print(df_res.to_string(index=False))


In [ ]:
# CELL 7: DECISION ENGINE RECONCILIATION & CODE DIFF
print('=' * 80)
print('CELL 7: DECISION ENGINE RECONCILIATION & CODE DIFF')
print('=' * 80)

live_promoted = [('cape', '7d'), ('kdci', '7d'), ('supramax', '7d'), ('supramax', '14d'), ('supramax', '30d')]
surviving_pairs = df_res[df_res['verdict'] == 'GENUINE SIGNAL'][['asset', 'horizon']].values.tolist()
surviving_tuples = [(r[0].lower(), r[1].lower()) for r in surviving_pairs]

print(f'Prior Live Promoted Registry (5 pairs): {live_promoted}')
print(f'New Genuine Signal Surviving Pairs ({len(surviving_tuples)} pairs): {surviving_tuples}')
excluded = [p for p in live_promoted if p not in surviving_tuples]
print(f'Excluded / De-promoted Pairs ({len(excluded)} pairs): {excluded}')
print('\n✅ Benchmark execution and reconciliation complete!')
